In [0]:
from pyspark.sql import functions as F

dem = spark.table("workspace.silver.demand_forecast_asof")
wind = spark.table("workspace.silver.wind_forecast_asof")
price = spark.table("workspace.silver.system_prices")

# The backbone: every half hour of every target day, taken from the demand forecast table,
# which covers all 744 target days without gaps
spine = dem.select(
    "settlement_date", "settlement_period", "start_time_utc",
    F.col("national_demand_mw").alias("fc_national_demand_mw"),
    F.col("transmission_demand_mw").alias("fc_transmission_demand_mw"),
    F.col("age_at_decision_hours").alias("fc_demand_age_h"),
    "decision_time_utc",
)

# Wind forecast is hourly: each half hour takes the value of the hour it falls in
wind_h = wind.select(F.col("start_time_utc").alias("wind_hour_utc"),
                     F.col("wind_generation_mw").alias("fc_wind_mw"))

g = (spine
     .withColumn("wind_hour_utc", F.date_trunc("hour", "start_time_utc"))
     .join(wind_h, "wind_hour_utc", "left")
     .drop("wind_hour_utc")
     .join(price.select("settlement_date", "settlement_period",
                        F.col("system_sell_price").alias("target_price")),
           ["settlement_date", "settlement_period"], "left"))

display(g.select(
    F.count("*").alias("rows"),
    F.countDistinct("settlement_date", "settlement_period").alias("unique_half_hours"),
    F.countDistinct("settlement_date").alias("days"),
    F.sum(F.col("fc_wind_mw").isNull().cast("int")).alias("missing_wind_fc"),
    F.sum(F.col("target_price").isNull().cast("int")).alias("missing_target"),
))
display(g.filter("target_price IS NULL")
         .groupBy("settlement_date").count().orderBy("settlement_date"))

rows,unique_half_hours,days,missing_wind_fc,missing_target
35712,35712,744,0,49


settlement_date,count
2026-10-13,1
2026-10-14,48


In [0]:
from pyspark.sql import functions as F

UK = "Europe/London"
LOC_WIND = ["wind_off_hornsea", "wind_off_dogger", "wind_off_eastanglia",
            "wind_off_seagreen", "wind_off_moray", "wind_on_scotland"]
LOC_SOLAR = ["solar_cambs", "solar_kent", "solar_somerset"]
LOC_TEMP = ["temp_london", "temp_birmingham", "temp_manchester"]

# Step 1 again: backbone, Elexon forecasts, answer
dem = spark.table("workspace.silver.demand_forecast_asof")
wind = spark.table("workspace.silver.wind_forecast_asof")
price = spark.table("workspace.silver.system_prices")
g = (dem.select("settlement_date", "settlement_period", "start_time_utc",
                F.col("national_demand_mw").alias("fc_national_demand_mw"),
                F.col("transmission_demand_mw").alias("fc_transmission_demand_mw"),
                F.col("age_at_decision_hours").alias("fc_demand_age_h"),
                "decision_time_utc")
     .withColumn("wind_hour_utc", F.date_trunc("hour", "start_time_utc"))
     .join(wind.select(F.col("start_time_utc").alias("wind_hour_utc"),
                       F.col("wind_generation_mw").alias("fc_wind_mw")), "wind_hour_utc", "left")
     .drop("wind_hour_utc")
     .join(price.select("settlement_date", "settlement_period",
                        F.col("system_sell_price").alias("target_price")),
           ["settlement_date", "settlement_period"], "left"))

# Weather: one row per target day and hour, one column per place and variable
s = spark.table("workspace.silver.weather_forecast_asof")
wide = (s.groupBy("target_date", "valid_time_utc")
         .pivot("location", LOC_WIND + LOC_SOLAR + LOC_TEMP)
         .agg(F.first("temperature_2m").alias("temp"), F.first("shortwave_radiation").alias("rad"),
              F.first("wind_speed_100m").alias("w100"), F.first("cloud_cover").alias("cloud")))

snapshots = wide.select(
    F.col("target_date").alias("settlement_date"), F.col("valid_time_utc").alias("snap_hour"),
    *[F.col(f"{l}_w100").alias(f"w100_{l}") for l in LOC_WIND],
    *[F.col(f"{l}_cloud").alias(f"cloud_{l}") for l in LOC_SOLAR],
    *[F.col(f"{l}_temp").alias(f"temp_{l}") for l in LOC_TEMP])
sunshine = wide.select(
    F.col("target_date").alias("settlement_date"), F.col("valid_time_utc").alias("rad_hour"),
    *[F.col(f"{l}_rad").alias(f"rad_{l}") for l in LOC_SOLAR])
age = s.groupBy(F.col("target_date").alias("settlement_date")).agg(
    F.max("run_age_at_decision_hours").alias("weather_run_age_h"))

def mean_of(cols):
    return sum(F.col(c) for c in cols) / len(cols)

g2 = (g
      .withColumn("snap_hour", F.date_trunc("hour", F.col("start_time_utc") + F.expr("INTERVAL 30 MINUTES")))
      .withColumn("rad_hour", F.date_trunc("hour", "start_time_utc") + F.expr("INTERVAL 1 HOUR"))
      .join(snapshots, ["settlement_date", "snap_hour"], "left")
      .join(sunshine, ["settlement_date", "rad_hour"], "left")
      .join(age, "settlement_date", "left")
      .drop("snap_hour", "rad_hour")
      .withColumn("w100_mean", mean_of([f"w100_{l}" for l in LOC_WIND]))
      .withColumn("rad_mean", mean_of([f"rad_{l}" for l in LOC_SOLAR]))
      .withColumn("cloud_mean", mean_of([f"cloud_{l}" for l in LOC_SOLAR]))
      .withColumn("temp_mean", mean_of([f"temp_{l}" for l in LOC_TEMP])))

(g2.write.format("delta").mode("overwrite").option("overwriteSchema", True)
   .saveAsTable("workspace.gold.price_features"))

t = spark.table("workspace.gold.price_features")
display(t.select(
    F.count("*").alias("rows"),
    F.countDistinct("settlement_date", "settlement_period").alias("unique_half_hours"),
    *[F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
      for c in ["w100_mean", "rad_mean", "cloud_mean", "temp_mean", "weather_run_age_h"]],
))
display(t.filter("rad_mean IS NULL OR w100_mean IS NULL")
         .groupBy("settlement_date").count().orderBy("settlement_date"))
# Alignment check: average sunshine by UK time of day, June only
display(t.filter(F.month("settlement_date") == 6)
         .groupBy(F.date_format(F.from_utc_timestamp("start_time_utc", UK), "HH:mm").alias("uk_time"))
         .agg(F.round(F.avg("rad_mean"), 0).alias("avg_sunshine_w_m2"))
         .orderBy("uk_time"))

rows,unique_half_hours,null_w100_mean,null_rad_mean,null_cloud_mean,null_temp_mean,null_weather_run_age_h
35712,35712,96,96,96,96,96


settlement_date,count
2025-08-08,48
2025-08-09,48


uk_time,avg_sunshine_w_m2
00:00,0.0
00:30,0.0
01:00,0.0
01:30,0.0
02:00,0.0
02:30,0.0
03:00,0.0
03:30,0.0
04:00,1.0
04:30,1.0


In [0]:
from pyspark.sql import functions as F

UK = "Europe/London"

def lateness(table, label):
    """For each settlement day X, the decision for target day X+2 is 10:00 UK on X+1.
    Count rows of day X that were published after that moment."""
    t = spark.table(f"workspace.silver.{table}")
    decision_for_x_plus_2 = F.to_utc_timestamp(
        F.to_timestamp(F.concat(F.date_add("settlement_date", 1).cast("string"),
                                F.lit(" 10:00:00"))), UK)
    late_h = (F.unix_timestamp("published_at_utc") - F.unix_timestamp(decision_for_x_plus_2)) / 3600
    return (t.select(F.lit(label).alias("source"), late_h.alias("late_h"))
             .agg(F.first("source").alias("source"),
                  F.count("*").alias("rows"),
                  F.sum((F.col("late_h") > 0).cast("int")).alias("published_after_decision"),
                  F.round(F.max("late_h"), 2).alias("worst_hours_late")))

display(lateness("system_prices", "system prices")
        .unionByName(lateness("demand_outturn", "demand outturn"))
        .unionByName(lateness("generation_by_fuel", "generation by fuel")))

source,rows,published_after_decision,worst_hours_late
system prices,35663,22011,2863.5
demand outturn,35664,0,-9.0
generation by fuel,713280,0,-9.0


In [0]:
from pyspark.sql import functions as F

UK = "Europe/London"
sp = spark.table("workspace.silver.system_prices")
decision = F.to_utc_timestamp(
    F.to_timestamp(F.concat(F.date_add("settlement_date", 1).cast("string"), F.lit(" 10:00:00"))), UK)
late = sp.withColumn("late_h", (F.unix_timestamp("published_at_utc") - F.unix_timestamp(decision)) / 3600)

# 1. How late, in buckets
display(late.withColumn("bucket",
        F.when(F.col("late_h") <= 0, "1. on time")
         .when(F.col("late_h") <= 24, "2. up to 1 day late")
         .when(F.col("late_h") <= 168, "3. 1 to 7 days late")
         .otherwise("4. more than 7 days late"))
        .groupBy("bucket").agg(F.count("*").alias("rows"),
                               F.min("settlement_date").alias("first_day"),
                               F.max("settlement_date").alias("last_day"))
        .orderBy("bucket"))

# 2. For the very late ones: on which dates were they published?
display(late.filter("late_h > 168")
        .groupBy(F.to_date("published_at_utc").alias("published_on"))
        .agg(F.count("*").alias("rows"),
             F.min("settlement_date").alias("for_days_from"),
             F.max("settlement_date").alias("for_days_to"))
        .orderBy(F.col("rows").desc()).limit(15))

bucket,rows,first_day,last_day
1. on time,13652,2024-10-01,2026-10-13
2. up to 1 day late,20937,2024-10-01,2026-10-12
3. 1 to 7 days late,18,2024-11-27,2026-06-01
4. more than 7 days late,1056,2025-08-13,2026-01-23


published_on,rows,for_days_from,for_days_to
2025-12-12,672,2025-08-16,2025-08-29
2025-11-24,144,2025-10-30,2025-11-01
2025-12-11,143,2025-08-13,2025-08-15
2025-11-25,96,2025-11-02,2025-11-03
2026-03-20,1,2026-01-23,2026-01-23


In [0]:
from pyspark.sql import functions as F

UK = "Europe/London"
g = spark.table("workspace.gold.price_features")
targets = g.select("settlement_date", "settlement_period", "decision_time_utc")

def lagged(src, value_col, days, name):
    """Value from the same settlement period `days` earlier, only if published before the decision."""
    s = src.select(F.col("settlement_date").alias("_src_date"),
                   F.col("settlement_period").alias("_src_period"),
                   F.col(value_col).alias("_value"),
                   F.col("published_at_utc").alias("_published"))
    j = (targets.join(s, (F.date_sub(targets.settlement_date, days) == s._src_date)
                          & (targets.settlement_period == s._src_period), "left"))
    return j.select("settlement_date", "settlement_period",
                    F.when(F.col("_published") <= F.col("decision_time_utc"), F.col("_value"))
                     .alias(name),
                    # how many values existed but were blanked because they came out too late
                    (F.col("_value").isNotNull() & (F.col("_published") > F.col("decision_time_utc")))
                     .cast("int").alias(f"_{name}_blanked"))

price = spark.table("workspace.silver.system_prices")
demand = spark.table("workspace.silver.demand_outturn")
fuel_wind = spark.table("workspace.silver.generation_by_fuel").filter("fuel_type = 'WIND'")

lags = [
    lagged(price, "system_sell_price", 3, "price_lag3"),
    lagged(price, "system_sell_price", 7, "price_lag7"),
    lagged(demand, "initial_demand_outturn", 2, "demand_lag2"),
    lagged(fuel_wind, "generation", 2, "wind_actual_lag2"),
]

# Daily summary of prices 3 days earlier, using only prices published before the decision
day3 = (targets.select("settlement_date", "decision_time_utc").distinct()
        .join(price.select(F.col("settlement_date").alias("_d"), "system_sell_price", "published_at_utc"),
              F.date_sub(F.col("settlement_date"), 3) == F.col("_d"), "left")
        .filter(F.col("published_at_utc") <= F.col("decision_time_utc"))
        .groupBy("settlement_date")
        .agg(F.avg("system_sell_price").alias("price_d3_mean"),
             F.min("system_sell_price").alias("price_d3_min"),
             F.max("system_sell_price").alias("price_d3_max"),
             F.count("*").alias("price_d3_count")))

local = F.from_utc_timestamp("start_time_utc", UK)
g3 = g
for lag in lags:
    g3 = g3.join(lag, ["settlement_date", "settlement_period"], "left")
g3 = (g3.join(day3, "settlement_date", "left")
        .withColumn("uk_hour", F.hour(local) + F.minute(local) / 60)
        .withColumn("day_of_week", F.dayofweek(local))            # 1 = Sunday, 7 = Saturday
        .withColumn("is_weekend", F.dayofweek(local).isin(1, 7).cast("int"))
        .withColumn("month", F.month(local)))

blanked = g3.select(*[F.sum(c).alias(c) for c in g3.columns if c.endswith("_blanked")])
g3 = g3.drop(*[c for c in g3.columns if c.endswith("_blanked")])

(g3.write.format("delta").mode("overwrite").option("overwriteSchema", True)
   .saveAsTable("workspace.gold.price_features"))

t = spark.table("workspace.gold.price_features")
print("rows:", t.count(), "| columns:", len(t.columns))
display(blanked)
display(t.select(*[F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
                   for c in ["price_lag3", "price_lag7", "demand_lag2", "wind_actual_lag2",
                             "price_d3_mean", "uk_hour"]]))

rows: 35712 | columns: 41


_price_lag3_blanked,_price_lag7_blanked,_demand_lag2_blanked,_wind_actual_lag2_blanked
1074,1056,0,0


null_price_lag3,null_price_lag7,null_demand_lag2,null_wind_actual_lag2,null_price_d3_mean,null_uk_hour
1226,1400,104,104,1152,0


In [0]:
from pyspark.sql import functions as F

t = spark.table("workspace.gold.price_features")
too_few = F.col("price_d3_count") < 43   # fewer than 90% of a 48 half hour day
t2 = (t.withColumn("price_d3_mean", F.when(too_few, None).otherwise(F.col("price_d3_mean")))
       .withColumn("price_d3_min", F.when(too_few, None).otherwise(F.col("price_d3_min")))
       .withColumn("price_d3_max", F.when(too_few, None).otherwise(F.col("price_d3_max"))))

(t2.write.format("delta").mode("overwrite").option("overwriteSchema", True)
   .saveAsTable("workspace.gold.price_features"))

t = spark.table("workspace.gold.price_features")
display(t.select(F.sum(F.col("price_d3_mean").isNull().cast("int")).alias("null_price_d3_mean")))
display(t.filter("price_d3_count IS NOT NULL AND price_d3_count < 43")
         .groupBy("settlement_date", "price_d3_count").count().orderBy("settlement_date"))

null_price_d3_mean
1296


settlement_date,price_d3_count,count
2024-11-30,39,48
2025-08-16,1,48
2026-06-04,41,48


In [0]:
from pyspark.sql import functions as F

t = spark.table("workspace.gold.price_features")
cols = [c for c in t.columns]
print(len(cols), "columns:", cols)

# 1. Overall shape
display(t.select(
    F.count("*").alias("rows"),
    F.sum(F.col("target_price").isNotNull().cast("int")).alias("training_rows"),
    F.sum(F.col("target_price").isNull().cast("int")).alias("rows_to_predict"),
))

# 2. The 10am rule: every decision is before its half hour, every forecast was published before it
display(t.select(
    F.sum((F.col("decision_time_utc") >= F.col("start_time_utc")).cast("int")).alias("decision_not_before_half_hour"),
    F.sum((F.col("fc_demand_age_h") < 0).cast("int")).alias("demand_fc_from_future"),
    F.sum((F.col("weather_run_age_h") < 6).cast("int")).alias("weather_run_too_fresh"),
    F.min("fc_demand_age_h").alias("min_demand_fc_age_h"),
    F.min("weather_run_age_h").alias("min_weather_age_h"),
))

# 3. Sanity: do the clues move with the price the way the market should?
checks = ["fc_national_demand_mw", "fc_wind_mw", "w100_mean", "rad_mean", "temp_mean",
          "price_lag3", "price_lag7", "price_d3_mean", "demand_lag2", "wind_actual_lag2"]
display(t.filter("target_price IS NOT NULL").select(
    *[F.round(F.corr("target_price", c), 3).alias(c) for c in checks]))

41 columns: ['settlement_date', 'settlement_period', 'start_time_utc', 'fc_national_demand_mw', 'fc_transmission_demand_mw', 'fc_demand_age_h', 'decision_time_utc', 'fc_wind_mw', 'target_price', 'w100_wind_off_hornsea', 'w100_wind_off_dogger', 'w100_wind_off_eastanglia', 'w100_wind_off_seagreen', 'w100_wind_off_moray', 'w100_wind_on_scotland', 'cloud_solar_cambs', 'cloud_solar_kent', 'cloud_solar_somerset', 'temp_temp_london', 'temp_temp_birmingham', 'temp_temp_manchester', 'rad_solar_cambs', 'rad_solar_kent', 'rad_solar_somerset', 'weather_run_age_h', 'w100_mean', 'rad_mean', 'cloud_mean', 'temp_mean', 'price_lag3', 'price_lag7', 'demand_lag2', 'wind_actual_lag2', 'price_d3_mean', 'price_d3_min', 'price_d3_max', 'price_d3_count', 'uk_hour', 'day_of_week', 'is_weekend', 'month']


rows,training_rows,rows_to_predict
35712,35663,49


decision_not_before_half_hour,demand_fc_from_future,weather_run_too_fresh,min_demand_fc_age_h,min_weather_age_h
0,0,0,0.05,9.0


fc_national_demand_mw,fc_wind_mw,w100_mean,rad_mean,temp_mean,price_lag3,price_lag7,price_d3_mean,demand_lag2,wind_actual_lag2
0.265,-0.232,-0.224,-0.131,-0.014,0.158,0.153,0.138,0.163,-0.024
